In [1]:
# Instalación de requisitos previos
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
requirements_path = REPO_ROOT / "requirements.txt"

%pip install -q -r {requirements_path}

print("Requisitos instalados correctamente.")


Note: you may need to restart the kernel to use updated packages.
Requisitos instalados correctamente.


In [2]:
# IMPORTS
import sys

from google.cloud import storage
from IPython.display import clear_output, display

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import (
    OUTPUT_DIR,
    FINAL_RELABEL_DIR,
    GCS_DIARIZATION_OUTPUT_PREFIX,
    GCS_FINAL_RELABEL_PREFIX,
    GCS_CONSOLIDATED_PREFIX,
    CONSOLIDATED_DIR,
    CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_CSV,
    CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_DEDUP_CSV,
    CONSOLIDATED_AUDIT_FINAL_MERGED_FILES_CSV,
    CONSOLIDATED_DROPPED_DUPLICATE_FILES_CSV,
    CONSOLIDATED_PREVIOUS_BACKUP_CSV,
    EXPECTED_AUDIOS_CONSOLIDATION,
    ensure_phase04_directories,
)

from src.storage_io import (
    download_directory,
    download_prefix_to_local,
    upload_directory,
)

from src.consolidacion_segmentos import (
    consolidation_outputs_exist,
    run_consolidation_pipeline,
)

print("Configuración y funciones importadas correctamente.")


Configuración y funciones importadas correctamente.


In [3]:
# CONFIGURACIÓN Y CLIENTE DE GCS

FORCE_CONSOLIDATION = False

ensure_phase04_directories()
gcs_client = storage.Client()

print("Directorio de inputs:", FINAL_RELABEL_DIR)
print("Directorio de outputs:", CONSOLIDATED_DIR)
print("Forzar reconstrucción:", FORCE_CONSOLIDATION)


Directorio de inputs: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/final_relabel
Directorio de outputs: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/consolidated
Forzar reconstrucción: False


In [4]:
# RESTAURAR OUTPUTS EXISTENTES E INPUTS NECESARIOS DESDE GCS

if not FORCE_CONSOLIDATION:
    download_directory(
        local_dir=CONSOLIDATED_DIR,
        gcs_prefix=GCS_DIARIZATION_OUTPUT_PREFIX,
        gcs_client=gcs_client,
        base_dir=OUTPUT_DIR,
        clear_output_fn=clear_output,
    )

outputs_available = consolidation_outputs_exist(
    CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_CSV,
    CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_DEDUP_CSV,
    CONSOLIDATED_AUDIT_FINAL_MERGED_FILES_CSV,
    CONSOLIDATED_DROPPED_DUPLICATE_FILES_CSV,
)

if FORCE_CONSOLIDATION or not outputs_available:
    download_prefix_to_local(
        source_uri=GCS_FINAL_RELABEL_PREFIX,
        local_dir=FINAL_RELABEL_DIR,
        gcs_client=gcs_client,
        suffix="_final_merged.csv",
        force=False,
        clear_output_fn=clear_output,
    )
else:
    print(
        "Inputs individuales no descargados porque los outputs "
        "consolidados ya están disponibles."
    )


Restauración desde GCS completada.
Archivos encontrados: 5
Archivos descargados: 0
Archivos locales ya vigentes: 5
Inputs individuales no descargados porque los outputs consolidados ya están disponibles.


In [5]:
# EJECUCIÓN O REUTILIZACIÓN DE LA CONSOLIDACIÓN

consolidation_outputs = run_consolidation_pipeline(
    final_dir=FINAL_RELABEL_DIR,
    output_path=CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_CSV,
    dedup_output_path=(
        CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_DEDUP_CSV
    ),
    audit_path=CONSOLIDATED_AUDIT_FINAL_MERGED_FILES_CSV,
    dropped_path=CONSOLIDATED_DROPPED_DUPLICATE_FILES_CSV,
    backup_path=CONSOLIDATED_PREVIOUS_BACKUP_CSV,
    force=FORCE_CONSOLIDATION,
)

audit_files = consolidation_outputs["audit_files"]
dupes_files = consolidation_outputs["dupes_files"]
canonical_files = consolidation_outputs["canonical_files"]
dropped_files = consolidation_outputs["dropped_files"]
df_segments = consolidation_outputs["df_segments"]
audit_segments = consolidation_outputs["audit_segments"]


Outputs consolidados encontrados. Se reutilizan sin reconstruir la consolidación.


In [6]:
# AUDITORÍA DE ARCHIVOS Y SELECCIÓN CANÓNICA

print("CSV *_final_merged encontrados:", audit_files["source_csv"].nunique())
print("Audio_id_base únicos normalizados:", audit_files["audio_id_base"].nunique())
print("Audio_id_base con más de un CSV:", len(dupes_files))

if len(dupes_files) > 0:
    display(dupes_files.head(50))

print("\nCSV originales:", len(audit_files))
print("CSV seleccionados:", len(canonical_files))
print(
    "CSV descartados por duplicidad de audio_id_base:",
    len(dropped_files),
)
print(
    "Audios reales normalizados seleccionados:",
    canonical_files["audio_id_base"].nunique(),
)

if canonical_files["audio_id_base"].nunique() != EXPECTED_AUDIOS_CONSOLIDATION:
    print(
        "⚠️ OJO: el número de audios normalizados no coincide con",
        EXPECTED_AUDIOS_CONSOLIDATION,
    )

if len(dropped_files) > 0:
    display(
        dropped_files[
            [
                "source_csv",
                "audio_id_base",
                "has_temp_prefix",
            ]
        ].head(50)
    )


CSV *_final_merged encontrados: 1181
Audio_id_base únicos normalizados: 1181
Audio_id_base con más de un CSV: 0

CSV originales: 1181
CSV seleccionados: 1181
CSV descartados por duplicidad de audio_id_base: 0
Audios reales normalizados seleccionados: 1181
⚠️ OJO: el número de audios normalizados no coincide con 1200


In [7]:
# RESUMEN Y AUDITORÍA DEL CONSOLIDADO

print("Dimensiones:", df_segments.shape)
print("audio_file únicos:", df_segments["audio_file"].nunique())
print("audio_id_base únicos:", df_segments["audio_id_base"].nunique())
print("Segmentos totales:", len(df_segments))
print("Columnas:", df_segments.columns.tolist())

display(df_segments.head())

print(
    "\nAudios normalizados en consolidado:",
    df_segments["audio_id_base"].nunique(),
)
print(
    "Audio_id_base con múltiples audio_file/source_csv "
    "después de consolidar:",
    len(audit_segments),
)

if len(audit_segments) > 0:
    display(audit_segments.head(50))
else:
    print("✅ Consolidado sin duplicidades aparentes por audio_id_base.")


Dimensiones: (40352, 26)
audio_file únicos: 1181
audio_id_base únicos: 1181
Segmentos totales: 40352
Columnas: ['segment_id_raw', 'audio_file', 'audio_stem', 'start', 'end', 'duration', 'speaker', 'rms_dbfs', 'overlap_duration_sec', 'overlap_ratio', 'valid_export', 'valid_anchor', 'drop_reasons', 'anchor_reasons', 'speaker_original', 'speaker_final', 'relabel_source', 'best_distance', 'second_distance', 'distance_margin', 'dist_SPEAKER_00', 'dist_SPEAKER_01', 'merged_n_segments', 'segment_ids_raw', 'source_csv', 'audio_id_base']


,segment_id_raw,audio_file,audio_stem,start,end,duration,speaker,rms_dbfs,overlap_duration_sec,overlap_ratio,...,relabel_source,best_distance,second_distance,distance_margin,dist_SPEAKER_00,dist_SPEAKER_01,merged_n_segments,segment_ids_raw,source_csv,audio_id_base
0,1,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,0.030969,4.452219,4.421250,SPEAKER_01,-28.919975,0.000000,0.000000,...,centroid,0.145629,0.747864,0.602235,0.747864,0.145629,1,1,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851
1,2,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,5.211594,6.342219,1.130625,SPEAKER_00,-33.883114,0.000000,0.000000,...,centroid,0.607482,0.814511,0.207029,0.607482,0.814511,1,2,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851
2,3,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,6.426594,13.800969,7.374375,SPEAKER_01,-29.186810,0.000000,0.000000,...,centroid,0.071406,0.732181,0.660775,0.732181,0.071406,1,3,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851
3,4,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,13.800969,14.965344,1.164375,SPEAKER_00,-36.389011,0.118125,0.101449,...,centroid,0.689627,0.975324,0.285697,0.689627,0.975324,1,4,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851
4,5,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,15.285969,19.994094,4.708125,SPEAKER_01,-27.753717,0.000000,0.000000,...,centroid,0.203869,0.709531,0.505662,0.709531,0.203869,1,5,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851



Audios normalizados en consolidado: 1181
Audio_id_base con múltiples audio_file/source_csv después de consolidar: 0
✅ Consolidado sin duplicidades aparentes por audio_id_base.


In [8]:
# RUTAS DE SALIDA Y RESUMEN FINAL

print("Archivo consolidado:", CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_CSV)
print(
    "Archivo duplicado de seguridad:",
    CONSOLIDATED_ALL_FINAL_MERGED_SEGMENTS_DEDUP_CSV,
)
print(
    "Auditoría de archivos:",
    CONSOLIDATED_AUDIT_FINAL_MERGED_FILES_CSV,
)
print(
    "Archivos descartados por duplicidad:",
    CONSOLIDATED_DROPPED_DUPLICATE_FILES_CSV,
)

if CONSOLIDATED_PREVIOUS_BACKUP_CSV.exists():
    print(
        "Backup anterior:",
        CONSOLIDATED_PREVIOUS_BACKUP_CSV,
    )

print("\nResumen final:")
print(
    "Audios únicos por audio_file:",
    df_segments["audio_file"].nunique(),
)
print(
    "Audios únicos por audio_id_base:",
    df_segments["audio_id_base"].nunique(),
)
print("Segmentos totales:", len(df_segments))


Archivo consolidado: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/consolidated/all_final_merged_segments.csv
Archivo duplicado de seguridad: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/consolidated/all_final_merged_segments_dedup.csv
Auditoría de archivos: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/consolidated/audit_final_merged_files.csv
Archivos descartados por duplicidad: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/consolidated/dropped_duplicate_final_merged_files.csv
Backup anterior: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/consolidated/all_final_merged_segments_previous_backup.csv

Resumen final:
Audios únicos por audio_file: 1181
Audios únicos por audio_id_base: 1181
Segmentos totales: 40352


In [9]:
# SUBIDA FINAL DE OUTPUTS NUEVOS O MODIFICADOS A GCS

upload_directory(
    local_dir=CONSOLIDATED_DIR,
    gcs_prefix=GCS_CONSOLIDATED_PREFIX,
    gcs_client=gcs_client,
    clear_output_fn=clear_output,
    skip_unchanged=True,
)


Subida final completada.
Archivos locales revisados: 5
Archivos subidos: 0
Archivos omitidos sin cambios: 5
Destino: gs://catedras_audio_detection/pipelineA/procesados_UNAV/diarization_outputs/consolidated/


{'total': 5, 'uploaded': 0, 'skipped': 5}